# Flipkart Reviews Sentiment Analysis

An end-to-end **NLP and Machine Learning** project for classifying Flipkart product reviews into **positive, neutral, and negative** sentiment categories.

### Objectives
- Explore and clean customer review data
- Convert ratings into sentiment labels
- Perform NLP preprocessing and exploratory data analysis
- Extract TF-IDF text features
- Handle class imbalance with SMOTE
- Compare multiple machine learning classifiers
- Evaluate models using Accuracy, Precision, Recall, F1-score, and ROC-AUC
- Generate additional sentiment insights using VADER

> **Dataset:** 2,304 raw Flipkart reviews. After removing duplicates, 2,181 unique reviews remain.


## 1. Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import string
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils import shuffle
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

RANDOM_STATE = 42


## 2. Load Dataset

In [ ]:
df = pd.read_csv("flipkart.csv")

print("Raw dataset shape:", df.shape)
display(df.head())


## 3. Data Cleaning and Exploratory Data Analysis

In [ ]:
print("Columns:", df.columns.tolist())
print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

# Remove the exported dataframe index column
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

# Remove duplicate records
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print("\nShape after duplicate removal:", df.shape)
df.info()


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Rating", order=sorted(df["Rating"].unique()))
plt.title("Distribution of Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")
plt.show()


## 4. Convert Ratings to Sentiment

In [ ]:
def map_rating_to_sentiment(rating):
    if rating in [1, 2]:
        return "negative"
    elif rating == 3:
        return "neutral"
    return "positive"

df["sentiment"] = df["Rating"].apply(map_rating_to_sentiment)

print("Sentiment distribution:")
display(df["sentiment"].value_counts())

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="sentiment", order=["positive", "neutral", "negative"])
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.show()


## 5. Text Preprocessing

In [ ]:
# Optional NLP dependencies used by the original project
# Install once in your environment if needed:
# pip install contractions emoji spacy
# python -m spacy download en_core_web_sm

import contractions
import emoji
import spacy

nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    text = str(text).lower()
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return contractions.fix(text)

def lemmatize_text(text):
    doc = nlp(text)
    return " ".join(token.lemma_ for token in doc)

df["clean_text"] = df["Review"].apply(clean_text)
df["clean_text"] = df["clean_text"].apply(lemmatize_text)

display(df[["Review", "clean_text", "sentiment"]].head())


## 6. Feature Engineering and TF-IDF

In [ ]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["sentiment"])

print("Label mapping:")
for label, value in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"{label} -> {value}")

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df["clean_text"])
y = df["label"]

print("TF-IDF matrix shape:", X.shape)


## 7. Train/Test Split

In [ ]:
# Keep the same split strategy used in the original analysis
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])


## 8. Baseline Model Comparison

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)

    return {
        "Model": name,
        "Test Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, average="weighted"),
        "Recall": recall_score(y_test, predictions, average="weighted"),
        "F1 Score": f1_score(y_test, predictions, average="weighted"),
        "ROC AUC": roc_auc_score(y_test, probabilities, multi_class="ovr")
    }

baseline_models = [
    (LogisticRegression(multi_class="multinomial", solver="lbfgs", random_state=RANDOM_STATE), "Logistic Regression"),
    (DecisionTreeClassifier(random_state=RANDOM_STATE), "Decision Tree"),
    (RandomForestClassifier(random_state=RANDOM_STATE), "Random Forest"),
    (MultinomialNB(), "Multinomial Naive Bayes"),
    (XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False, eval_metric="mlogloss"), "XGBoost")
]

baseline_results = pd.DataFrame([
    evaluate_model(model, X_train, y_train, X_test, y_test, name)
    for model, name in baseline_models
])

baseline_results.sort_values("Test Accuracy", ascending=False)


## 9. Handle Class Imbalance with SMOTE

In [ ]:
print("Class distribution before SMOTE:")
print(pd.Series(y_train).value_counts().sort_index())

smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\nClass distribution after SMOTE:")
print(pd.Series(y_train_res).value_counts().sort_index())


## 10. Model Comparison After Balancing

In [ ]:
balanced_models = [
    (LogisticRegression(multi_class="multinomial", solver="lbfgs", random_state=RANDOM_STATE), "Logistic Regression"),
    (DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE), "Decision Tree"),
    (RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE), "Random Forest"),
    (MultinomialNB(), "Multinomial Naive Bayes"),
    (XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=RANDOM_STATE), "XGBoost")
]

balanced_results = pd.DataFrame([
    evaluate_model(model, X_train_res, y_train_res, X_test, y_test, name)
    for model, name in balanced_models
])

balanced_results.sort_values("Test Accuracy", ascending=False)


## 11. XGBoost Detailed Evaluation

In [ ]:
xgb_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=RANDOM_STATE
)

xgb_model.fit(X_train_res, y_train_res)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)

xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb, average="weighted")
xgb_recall = recall_score(y_test, y_pred_xgb, average="weighted")
xgb_f1 = f1_score(y_test, y_pred_xgb, average="weighted")
xgb_roc_auc = roc_auc_score(y_test, y_proba_xgb, multi_class="ovr")

print(f"Accuracy : {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall   : {xgb_recall:.4f}")
print(f"F1 Score : {xgb_f1:.4f}")
print(f"ROC AUC  : {xgb_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_xgb,
    target_names=label_encoder.classes_
))


In [ ]:
cm = confusion_matrix(y_test, y_pred_xgb)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## 12. VADER Sentiment Analysis

VADER provides an additional rule-based sentiment signal that can be compared with the supervised ML classification approach.


In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download("vader_lexicon", quiet=True)

sia = SentimentIntensityAnalyzer()

df["Positive"] = df["clean_text"].apply(lambda x: sia.polarity_scores(x)["pos"])
df["Negative"] = df["clean_text"].apply(lambda x: sia.polarity_scores(x)["neg"])
df["Neutral"] = df["clean_text"].apply(lambda x: sia.polarity_scores(x)["neu"])

vader_totals = {
    "Positive": df["Positive"].sum(),
    "Negative": df["Negative"].sum(),
    "Neutral": df["Neutral"].sum()
}

vader_totals


## 13. Business Insights

The analysis can support product and customer-experience teams by:
- identifying recurring negative feedback,
- monitoring overall customer sentiment,
- comparing sentiment patterns across products,
- prioritizing reviews that require attention,
- reducing manual review-classification effort.

### Key Result from the original analysis

The original project evaluation reported **XGBoost as the best-performing model after balancing**, with:
- **Test Accuracy: 92.67%**
- **Weighted F1 Score: 93.14%**
- **ROC-AUC: 96.14%**

These are the metrics used in the project resume entry.


## 14. Conclusion

This project demonstrates an end-to-end NLP workflow:

**Raw Reviews → Cleaning → Sentiment Labels → TF-IDF → Class Imbalance Handling → ML Models → Evaluation → Business Insights**

The project combines classical NLP techniques with supervised machine learning to transform unstructured customer reviews into actionable sentiment information.
